<a href="https://colab.research.google.com/github/ThayseBachmeyer/ProcessamentoSinaisI/blob/main/Aula%2006/C%C3%B3digos/prat6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Código utilizado na questão 1-a

In [ ]:
#n°1-a

import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import hadamard
from scipy.fft import fft, fftshift, fftfreq

np.random.seed(42)
N_bits = 8
m1 = np.random.choice([-1, 1], size=N_bits)
m2 = np.random.choice([-1, 1], size=N_bits)
m3 = np.random.choice([-1, 1], size=N_bits)

H8 = hadamard(8)
c1 = H8[1, :]
c2 = H8[2, :]
c3 = H8[3, :]

s1 = np.kron(m1, c1)
s2 = np.kron(m2, c2)
s3 = np.kron(m3, c3)


fig_tempo, axs_tempo = plt.subplots(3, 3, figsize=(16, 9))
usuarios = [(m1, c1, s1), (m2, c2, s2), (m3, c3, s3)]

for i, (m, c, s) in enumerate(usuarios):
    axs_tempo[i, 0].stem(m, basefmt=" ", linefmt='b-', markerfmt='bo')
    axs_tempo[i, 0].set_title(f'Usuário {i+1}: Sinal Original $m_{i+1}[n]$')
    axs_tempo[i, 0].set_ylim(-1.5, 1.5)
    axs_tempo[i, 0].grid(True, alpha=0.3)

    axs_tempo[i, 1].stem(c, basefmt=" ", linefmt='r-', markerfmt='ro')
    axs_tempo[i, 1].set_title(f'Usuário {i+1}: Código $c_{i+1}[n]$')
    axs_tempo[i, 1].set_ylim(-1.5, 1.5)
    axs_tempo[i, 1].grid(True, alpha=0.3)

    axs_tempo[i, 2].step(range(len(s)), s, where='mid', color='purple')
    axs_tempo[i, 2].set_title(f'Usuário {i+1}: Sinal Codificado (Espalhado)')
    axs_tempo[i, 2].set_ylim(-1.5, 1.5)
    axs_tempo[i, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


fig_freq, axs_freq = plt.subplots(3, 2, figsize=(14, 9))

for i, (m, c, s) in enumerate(usuarios):
    m_upsampled = np.repeat(m, 8)

    N_fft = 512
    M_espectro = np.abs(fftshift(fft(m_upsampled, n=N_fft)))
    S_espectro = np.abs(fftshift(fft(s, n=N_fft)))

    f = fftshift(fftfreq(N_fft))

    axs_freq[i, 0].plot(f, M_espectro, 'b', linewidth=2)
    axs_freq[i, 0].set_title(f'Usuário {i+1}: Espectro do Sinal Original')
    axs_freq[i, 0].set_xlabel('Frequência Normalizada')
    axs_freq[i, 0].grid(True, alpha=0.4)
    axs_freq[i, 0].set_xlim(-0.5, 0.5)

    axs_freq[i, 1].plot(f, S_espectro, 'purple', linewidth=2)
    axs_freq[i, 1].set_title(f'Usuário {i+1}: Espectro do Sinal Codificado')
    axs_freq[i, 1].set_xlabel('Frequência Normalizada')
    axs_freq[i, 1].grid(True, alpha=0.4)
    axs_freq[i, 1].set_xlim(-0.5, 0.5)

plt.tight_layout()
plt.show()

## Código utilizado na questão 1-b

In [ ]:
#n°1-b

import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import hadamard
from scipy.fft import fft, fftshift, fftfreq

np.random.seed(42)
m1 = np.random.choice([-1, 1], size=8)
m2 = np.random.choice([-1, 1], size=8)
m3 = np.random.choice([-1, 1], size=8)

H8 = hadamard(8)
c1 = H8[1, :]
c2 = H8[2, :]
c3 = H8[3, :]

x1 = np.kron(m1, c1)
x2 = np.kron(m2, c2)
x3 = np.kron(m3, c3)

x_sum = x1 + x2 + x3
variancias = [1e-3, 1e-2, 1e-1, 1]

fig_tempo, axs_tempo = plt.subplots(2, 2, figsize=(14, 10))
fig_freq, axs_freq = plt.subplots(2, 2, figsize=(14, 10))
axs_tempo = axs_tempo.flatten()
axs_freq = axs_freq.flatten()

N_chips = len(x_sum)
N_fft = 512
f = fftshift(fftfreq(N_fft))


for i, var in enumerate(variancias):
    sigma = np.sqrt(var)
    w = np.random.normal(0, sigma, size=N_chips)
    y = x_sum + w

    axs_tempo[i].step(range(N_chips), y, where='mid', color='teal', linewidth=1.5)
    axs_tempo[i].set_title(f'Sinal Recebido $y[n]$ para $\sigma^2 = {var}$')
    axs_tempo[i].set_xlabel('Amostras (Chips)')
    axs_tempo[i].set_ylabel('Amplitude')
    axs_tempo[i].grid(True, alpha=0.4)
    axs_tempo[i].set_ylim(-5, 5)
    Y_espectro = np.abs(fftshift(fft(y, n=N_fft)))

    axs_freq[i].plot(f, Y_espectro, color='darkorange', linewidth=1.5)
    axs_freq[i].set_title(f'Espectro de $y[n]$ para $\sigma^2 = {var}$')
    axs_freq[i].set_xlabel('Frequência Normalizada')
    axs_freq[i].set_ylabel('Magnitude')
    axs_freq[i].grid(True, alpha=0.4)
    axs_freq[i].set_xlim(-0.5, 0.5)

fig_tempo.tight_layout()
fig_tempo.suptitle('Domínio do Tempo: Sinal Recebido $y[n]$', fontsize=16, y=1.03)
fig_tempo.show()

fig_freq.tight_layout()
fig_freq.suptitle('Domínio da Frequência: Espectro de $y[n]$', fontsize=16, y=1.03)
fig_freq.show()

## Código utilizado na questão 1-d

In [ ]:
#n°1-d

import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import hadamard
from scipy.special import erfc

N = 8
N_bits = 10000
np.random.seed(42)
H8 = hadamard(N)
c1 = H8[1, :]
c2 = H8[2, :]
c3 = H8[3, :]

variancias = np.logspace(-1, 1.5, 20)
prob_erro_simulada = []
prob_erro_teorica = []

for var in variancias:
    sigma = np.sqrt(var)
    erros = 0

    for _ in range(N_bits):
        m1 = np.random.choice([-1, 1])
        m2 = np.random.choice([-1, 1])
        m3 = np.random.choice([-1, 1])

        x1 = m1 * c1
        x2 = m2 * c2
        x3 = m3 * c3

        w = np.random.normal(0, sigma, size=N)
        y = x1 + x2 + x3 + w
        estimativa_m1 = (1/N) * np.dot(c1, y)
        bit_recuperado = 1 if estimativa_m1 > 0 else -1

        if bit_recuperado != m1:
            erros += 1

    ber = erros / N_bits
    prob_erro_simulada.append(ber)

    q_arg = np.sqrt(N / var)
    pe_teorica = 0.5 * erfc(q_arg / np.sqrt(2))
    prob_erro_teorica.append(pe_teorica)

plt.figure(figsize=(10, 6))
plt.plot(variancias, prob_erro_simulada, 'bo', label='Simulação Monte Carlo')
plt.plot(variancias, prob_erro_teorica, 'r-', linewidth=2, label='Curva Teórica $Q(\sqrt{N/\sigma^2})$')

plt.xscale('log')
plt.yscale('log')
plt.title('Probabilidade de Erro de Bit (BER) vs Variância do Ruído ($\sigma^2$)')
plt.xlabel('Variância do Ruído $\sigma^2$')
plt.ylabel('Probabilidade de Erro $P_e$')
plt.grid(True, which="both", ls="--", alpha=0.5)
plt.legend()
plt.ylim(1e-4, 1)

plt.axvline(x=1, color='gray', linestyle=':', label='$\sigma^2 = 1$')
plt.axvline(x=10**-1, color='green', linestyle=':', label='$\sigma^2 = 10^{-1}$')

plt.tight_layout()
plt.show()

## Código utilizado na questão 1-f



In [ ]:
#n°1-f

import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import hadamard

N = 8
C = hadamard(N)
c1 = C[1, :]
m1 = np.random.choice([-1, 1])
y = m1 * c1

variancias = np.logspace(-1, 1.5, 20)
prob_erro_simulada = []
N_bits = 100000
np.random.seed(42)

for var in variancias:
    sigma = np.sqrt(var)
    erros = 0

    for _ in range(N_bits):
        m_k = np.random.choice([-1, 1])
        y_n = m_k * c1
        W = np.random.normal(0, sigma, size=(N, N))

        C_hat = C + W
        c_k_hat = C_hat[1, :]
        m_k_estimado = (1/N) * np.dot(c_k_hat, y_n)
        bit_decidido = 1 if m_k_estimado > 0 else -1

        if bit_decidido != m_k:
            erros += 1

    ber = erros / N_bits
    prob_erro_simulada.append(ber)

plt.figure(figsize=(10, 6))

plt.plot(variancias, prob_erro_simulada, 'mo-', linewidth=2, label='Simulação Empírica ($\hat{\mathbf{C}} = \mathbf{C} + \mathbf{W}$)', color='green')

plt.xscale('log')
plt.yscale('log')
plt.title('BER vs Distorção na Matriz do Recetor ($\sigma^2$)')
plt.xlabel('Variância da Distorção da Matriz $\sigma^2$')
plt.ylabel('Probabilidade de Erro $P_e$')
plt.grid(True, which="both", ls="--", alpha=0.5)
plt.legend()
plt.ylim(1e-4, 1)

plt.tight_layout()
plt.show()

## Código utilizado na questão 3-b

In [ ]:
#n°3-b

import numpy as np
import pywt
import matplotlib.pyplot as plt

N = 2000
t = np.arange(N + 1)
x = np.zeros_like(t, dtype=np.float64)

pos_sin = np.array([0, 600, 1080, 1380, 1680, 2000])
T_sin = np.array([100, 40, 20, 10, 5])
for i in range(5):
    m, n = pos_sin[i], pos_sin[i+1]
    t_seg = np.arange(1, n - m + 1)
    x[m:n] = np.sin(2 * np.pi * t_seg / T_sin[i])

amp_imp = np.array([3, -2, 2, 2.5, -2.5])
pos_imp = np.array([200, 372, 1324, 1343, 1802])
T_imp = np.array([5, 25, 5, 5, 5])
for i in range(5):
    m = pos_imp[i]
    offset = int(T_imp[i] / 2)
    n = pos_imp[i] + offset
    t_seg = np.arange(1, n - m + 2)
    x[m:n+1] += amp_imp[i] * (np.sin(2 * np.pi * t_seg / T_imp[i]))**2


wavelet_name = 'bior4.4'
wavelet = pywt.Wavelet(wavelet_name)
cA, cD = pywt.dwt(x, wavelet)

dec_lo = wavelet.dec_lo
dec_hi = wavelet.dec_hi
rec_lo = wavelet.rec_lo
rec_hi = wavelet.rec_hi

print(f"--- Coeficientes dos Filtros da Wavelet '{wavelet_name}' ---")
print(f"Filtro de Análise (Passa-Baixo / dec_lo):\n {np.round(dec_lo, 5)}")
print(f"Filtro de Análise (Passa-Alto / dec_hi):\n {np.round(dec_hi, 5)}")
print(f"Filtro de Síntese (Passa-Baixo / rec_lo):\n {np.round(rec_lo, 5)}")
print(f"Filtro de Síntese (Passa-Alto / rec_hi):\n {np.round(rec_hi, 5)}")


fig, axs = plt.subplots(3, 1, figsize=(10, 8))

axs[0].plot(x, color='teal')
axs[0].set_title('Sinal Original $x(t)$ (Não Estacionário)')
axs[0].grid(True)

axs[1].plot(cA, color='darkorange')
axs[1].set_title(f'Coeficientes de Aproximação (cA) - Wavelet {wavelet_name}')
axs[1].grid(True)

axs[2].plot(cD, color='purple')
axs[2].set_title(f'Coeficientes de Detalhe (cD) - Wavelet {wavelet_name}')
axs[2].grid(True)

plt.tight_layout()
plt.show()

## Código utilizado na questão 3-c

In [ ]:
#n°3-c

import numpy as np
import pywt
import matplotlib.pyplot as plt

N = 2000
t = np.arange(N + 1)
x = np.zeros_like(t, dtype=np.float64)

pos_sin = np.array([0, 600, 1080, 1380, 1680, 2000])
T_sin = np.array([100, 40, 20, 10, 5])
for i in range(5):
    m, n = pos_sin[i], pos_sin[i+1]
    t_seg = np.arange(1, n - m + 1)
    x[m:n] = np.sin(2 * np.pi * t_seg / T_sin[i])

amp_imp = np.array([3, -2, 2, 2.5, -2.5])
pos_imp = np.array([200, 372, 1324, 1343, 1802])
T_imp = np.array([5, 25, 5, 5, 5])
for i in range(5):
    m = pos_imp[i]
    offset = int(T_imp[i] / 2)
    n = pos_imp[i] + offset
    t_seg = np.arange(1, n - m + 2)
    x[m:n+1] += amp_imp[i] * (np.sin(2 * np.pi * t_seg / T_imp[i]))**2


wavelet_name = 'bior4.4'
coeficientes = pywt.wavedec(x, wavelet_name, level=5)
cA5 = coeficientes[0]
detalhes = coeficientes[1:]
detalhes_invertidos = detalhes[::-1]


fig, axs = plt.subplots(7, 1, figsize=(10, 18))

axs[0].plot(t, x, color='blue')
axs[0].set_title('Sinal Original $x(t)$')
axs[0].set_ylabel('Amplitude')
axs[0].grid(True, alpha=0.4)
axs[0].set_xlabel('Tempo Normalizado')

cores = ['purple', 'teal', 'darkorange', 'brown', 'magenta']
nomes_detalhes = ['$cD_1$', '$cD_2$', '$cD_3$', '$cD_4$', '$cD_5$']
titulos_detalhes = ['Detalhe 1: Componentes de Frequência Muito Alta (Deteção de Transientes)', 'Detalhe 2: Componentes de Frequência Alta', 'Detalhe 3: Componentes de Frequência Média-Alta', 'Detalhe 4: Componentes de Frequência Média', 'Detalhe 5: Componentes de Frequência Média-Baixa']

for i in range(5):
    coef = detalhes_invertidos[i]
    t_norm = np.linspace(0, N, len(coef))

    axs[i+1].plot(t_norm, coef, color=cores[i])
    axs[i+1].set_title(f'Coeficientes de Detalhe - Estágio {i+1} ({nomes_detalhes[i]})', fontsize=12)
    axs[i+1].set_ylabel(nomes_detalhes[i])
    axs[i+1].set_xlabel('Tempo Normalizado')
    axs[i+1].grid(True, alpha=0.4)

t_norm_ca = np.linspace(0, N, len(cA5))
axs[6].plot(t_norm_ca, cA5, color='red', linewidth=2)
axs[6].set_title('Coeficientes de Aproximação - Estágio 5 ($cA_5$)')
axs[6].set_xlabel('Tempo Normalizado')
axs[6].set_ylabel('$cA_5$')
axs[6].grid(True, alpha=0.4)

plt.tight_layout()
plt.show()

## Código utilizado na questão 4-a

In [ ]:
#n°4-a

import numpy as np
import scipy.io as sio
import matplotlib.pyplot as plt
from scipy.fft import fft, fftfreq
import urllib.request
import os

url = "https://github.com/ThayseBachmeyer/ProcessamentoSinaisI/raw/refs/heads/main/Aula%2006/Dados/leleccum.mat"
filename = "leleccum.mat"

if not os.path.exists(filename):
    urllib.request.urlretrieve(url, filename)

mat_data = sio.loadmat(filename)
sinal = mat_data['leleccum'].flatten()
N = len(sinal)
t = np.arange(N)

espectro = np.abs(fft(sinal))
freqs = fftfreq(N)
metade = N // 2
freqs_positivas = freqs[:metade]
espectro_positivo = espectro[:metade]


fig, axs = plt.subplots(2, 1, figsize=(10, 8))

axs[0].plot(t, sinal, color='teal', linewidth=1)
axs[0].set_title('Sinal leleccum.mat Original no Tempo')
axs[0].set_xlabel('Amostras')
axs[0].set_ylabel('Amplitude')
axs[0].grid(True, alpha=0.4)

axs[1].plot(freqs_positivas, espectro_positivo, color='darkorange', linewidth=1)
axs[1].set_title('Espectro de Frequência do Sinal (Magnitude)')
axs[1].set_xlabel('Frequência Normalizada')
axs[1].set_ylabel('Magnitude')
axs[1].grid(True, alpha=0.4)

plt.tight_layout()
plt.show()

## Código utilizado na questão 4-b

In [ ]:
#n°4-b

import numpy as np
import pywt
import matplotlib.pyplot as plt

N = 2000
t = np.arange(N + 1)
x = np.zeros_like(t, dtype=np.float64)

pos_sin = np.array([0, 600, 1080, 1380, 1680, 2000])
T_sin = np.array([100, 40, 20, 10, 5])
for i in range(5):
    m, n = pos_sin[i], pos_sin[i+1]
    t_seg = np.arange(1, n - m + 1)
    x[m:n] = np.sin(2 * np.pi * t_seg / T_sin[i])

amp_imp = np.array([3, -2, 2, 2.5, -2.5])
pos_imp = np.array([200, 372, 1324, 1343, 1802])
T_imp = np.array([5, 25, 5, 5, 5])
for i in range(5):
    m = pos_imp[i]
    offset = int(T_imp[i] / 2)
    n = pos_imp[i] + offset
    t_seg = np.arange(1, n - m + 2)
    x[m:n+1] += amp_imp[i] * (np.sin(2 * np.pi * t_seg / T_imp[i]))**2


wavelet_name = 'db4'
wavelet = pywt.Wavelet(wavelet_name)

dec_lo = wavelet.dec_lo
dec_hi = wavelet.dec_hi
rec_lo = wavelet.rec_lo
rec_hi = wavelet.rec_hi

print(f"--- Coeficientes dos Filtros da Wavelet '{wavelet_name}' ---")
print(f"Filtro de Análise (Passa-Baixo):\n {np.round(dec_lo, 5)}")
print(f"Filtro de Análise (Passa-Alto):\n {np.round(dec_hi, 5)}")
print(f"Filtro de Síntese (Passa-Baixo):\n {np.round(rec_lo, 5)}")
print(f"Filtro de Síntese (Passa-Alto):\n {np.round(rec_hi, 5)}\n")

cA, cD = pywt.dwt(x, wavelet)


fig, axs = plt.subplots(3, 1, figsize=(10, 8))

axs[0].plot(x, color='teal')
axs[0].set_title('Sinal Original $x(t)$ (Não Estacionário)')
axs[0].grid(True)

axs[1].plot(cA, color='darkorange')
axs[1].set_title(f'Coeficientes de Aproximação (cA) - Wavelet {wavelet_name}')
axs[1].grid(True)

axs[2].plot(cD, color='purple')
axs[2].set_title(f'Coeficientes de Detalhe (cD) - Wavelet {wavelet_name}')
axs[2].grid(True)

plt.tight_layout()
plt.show()

## Código utilizado na questão 4-c

In [ ]:
#n°4-c

import numpy as np
import pywt
import matplotlib.pyplot as plt

N = 2000
t = np.arange(N + 1)
x = np.zeros_like(t, dtype=np.float64)

pos_sin = np.array([0, 600, 1080, 1380, 1680, 2000])
T_sin = np.array([100, 40, 20, 10, 5])
for i in range(5):
    m, n = pos_sin[i], pos_sin[i+1]
    t_seg = np.arange(1, n - m + 1)
    x[m:n] = np.sin(2 * np.pi * t_seg / T_sin[i])

amp_imp = np.array([3, -2, 2, 2.5, -2.5])
pos_imp = np.array([200, 372, 1324, 1343, 1802])
T_imp = np.array([5, 25, 5, 5, 5])
for i in range(5):
    m = pos_imp[i]
    offset = int(T_imp[i] / 2)
    n = pos_imp[i] + offset
    t_seg = np.arange(1, n - m + 2)
    x[m:n+1] += amp_imp[i] * (np.sin(2 * np.pi * t_seg / T_imp[i]))**2

wavelet_name = 'db4'
coeficientes = pywt.wavedec(x, wavelet_name, level=5)
cA5 = coeficientes[0]
cD_list = coeficientes[1:]
cD_list_invertida = cD_list[::-1]

cA_list = []
sinal_atual = x
for i in range(1, 6):
    cA, _ = pywt.dwt(sinal_atual, wavelet_name)
    cA_list.append(cA)
    sinal_atual = cA

fig, axs = plt.subplots(6, 2, figsize=(18, 16))

axs[0, 0].plot(t, x, color='black')
axs[0, 0].set_title('Sinal Original $x(t)$')
axs[0, 0].grid(True, alpha=0.4)
axs[0, 1].axis('off')

cores_detalhes = ['purple', 'teal', 'darkorange', 'brown', 'magenta']
cores_aprox = ['blue', 'cornflowerblue', 'royalblue', 'navy', 'darkblue']

for i in range(5):
    coef_d = cD_list_invertida[i]
    t_norm_d = np.linspace(0, N, len(coef_d))
    axs[i+1, 0].plot(t_norm_d, coef_d, color=cores_detalhes[i])
    axs[i+1, 0].set_ylabel(f'$cD_{i+1}$ (Detalhe)')
    axs[i+1, 0].grid(True, alpha=0.4)

    coef_a = cA_list[i]
    t_norm_a = np.linspace(0, N, len(coef_a))
    axs[i+1, 1].plot(t_norm_a, coef_a, color=cores_aprox[i])
    axs[i+1, 1].set_ylabel(f'$cA_{i+1}$ (Aproximação)')
    axs[i+1, 1].grid(True, alpha=0.4)

axs[5, 0].set_xlabel('Tempo Normalizado')
axs[5, 1].set_xlabel('Tempo Normalizado')

plt.suptitle('Transformada de Wavelet (Daubechies-4) de 5 Estágios', fontsize=16)
plt.tight_layout(rect=[0, 0.03, 1, 0.97])
plt.show()

## Código utilizado na questão 4-d

In [ ]:
#n°4-d

import numpy as np
import pywt
import matplotlib.pyplot as plt

N = 2000
t = np.arange(N + 1)
x = np.zeros_like(t, dtype=np.float64)

pos_sin = np.array([0, 600, 1080, 1380, 1680, 2000])
T_sin = np.array([100, 40, 20, 10, 5])
for i in range(5):
    m, n = pos_sin[i], pos_sin[i+1]
    t_seg = np.arange(1, n - m + 1)
    x[m:n] = np.sin(2 * np.pi * t_seg / T_sin[i])

amp_imp = np.array([3, -2, 2, 2.5, -2.5])
pos_imp = np.array([200, 372, 1324, 1343, 1802])
T_imp = np.array([5, 25, 5, 5, 5])
for i in range(5):
    m = pos_imp[i]
    offset = int(T_imp[i] / 2)
    n = pos_imp[i] + offset
    t_seg = np.arange(1, n - m + 2)
    x[m:n+1] += amp_imp[i] * (np.sin(2 * np.pi * t_seg / T_imp[i]))**2

wavelet_name = 'db4'
niveis = 5
limiares_teste = [0.1, 0.5, 1.5]
sinais_reconstruidos = []

for limiar in limiares_teste:
    coeficientes = pywt.wavedec(x, wavelet_name, level=niveis)
    coeficientes_limiarizados = []
    cA_limiar = pywt.threshold(coeficientes[0], value=limiar, mode='soft')
    coeficientes_limiarizados.append(cA_limiar)

    for cD in coeficientes[1:]:
        cD_limiar = pywt.threshold(cD, value=limiar, mode='soft')
        coeficientes_limiarizados.append(cD_limiar)

    x_rec = pywt.waverec(coeficientes_limiarizados, wavelet_name)
    sinais_reconstruidos.append(x_rec[:len(x)])

fig, axs = plt.subplots(4, 1, figsize=(10, 8))

axs[0].plot(t, x, color='blue')
axs[0].set_title('Sinal Original (Limiar = 0)')
axs[0].grid(True, alpha=0.4)

cores = ['green', 'orange', 'red']
for i, limiar in enumerate(limiares_teste):
    axs[i+1].plot(t, sinais_reconstruidos[i], color=cores[i])
    axs[i+1].set_title(f'Sinal Reconstruído com Limiar Soft = {limiar}')
    axs[i+1].grid(True, alpha=0.4)

axs[3].set_xlabel('Amostras (Tempo)')
plt.tight_layout()
plt.show()

## Código utilizado na questão 4-e

In [ ]:
#n°4-e

import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import fft, ifft, fftfreq

N = 2000
t = np.arange(N + 1)
x = np.zeros_like(t, dtype=np.float64)

pos_sin = np.array([0, 600, 1080, 1380, 1680, 2000])
T_sin = np.array([100, 40, 20, 10, 5])
for i in range(5):
    m, n = pos_sin[i], pos_sin[i+1]
    t_seg = np.arange(1, n - m + 1)
    x[m:n] = np.sin(2 * np.pi * t_seg / T_sin[i])

amp_imp = np.array([3, -2, 2, 2.5, -2.5])
pos_imp = np.array([200, 372, 1324, 1343, 1802])
T_imp = np.array([5, 25, 5, 5, 5])
for i in range(5):
    m = pos_imp[i]
    offset = int(T_imp[i] / 2)
    n = pos_imp[i] + offset
    t_seg = np.arange(1, n - m + 2)
    x[m:n+1] += amp_imp[i] * (np.sin(2 * np.pi * t_seg / T_imp[i]))**2

X_dft = fft(x)
magnitude_dft = np.abs(X_dft)
limiares_teste_dft = [10, 50, 100]
sinais_reconstruidos_dft = []

for limiar in limiares_teste_dft:
    X_dft_limiarizado = X_dft.copy()
    X_dft_limiarizado[magnitude_dft < limiar] = 0

    x_rec_dft = np.real(ifft(X_dft_limiarizado))
    sinais_reconstruidos_dft.append(x_rec_dft)


fig, axs = plt.subplots(4, 1, figsize=(10, 8))

axs[0].plot(t, x, color='black')
axs[0].set_title('Sinal Original $x(t)$ (Limiar = 0)')
axs[0].grid(True, alpha=0.4)

cores = ['green', 'orange', 'red']

for i, limiar in enumerate(limiares_teste_dft):
    axs[i+1].plot(t, sinais_reconstruidos_dft[i], color=cores[i])
    axs[i+1].set_title(f'Sinal Reconstruído via DFT com Limiar = {limiar}')
    axs[i+1].grid(True, alpha=0.4)

axs[3].set_xlabel('Amostras (Tempo)')
plt.tight_layout()
plt.show()